In [ ]:
import xarray as xr    
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
path = '/work/bb1555/user/kolja/data/verdant/c3sar1-20240806-exp023_LWC_MYSTIC_TOBAC_combined.nc' # first dataset Kolja prepared for Vedant

ds = xr.load_dataset(path)


In [ ]:
ds


`..._std` likely means estimated standard deviation and quantifies noise for every grid cell, time, and experiment. (we have statistica noise because MYSTIC only uses a finite number of photons), (verify with kolja...)

In [ ]:
ds["masks"].plot()
plt.yscale("log")

## First plots

In [ ]:
ds["eglo"].sel(exp="1D")

In [ ]:
experiments = ["1D", "3D", "3D-1D", "1DCRE", "3DCRE", "CS"]

# Select dir/dif/global radiation field and one time step and remove the phy dimension of length 1
field = ds["eglo"].isel(time=52).squeeze()

fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)

for ax, exp in zip(axes.flat, experiments):
    data = field.sel(exp=exp)

    # Difference/CRE fields: diverging colour scale centred at zero
    if exp in ["3D-1D", "3DCRE", "1DCRE"]:
        limit = float(np.nanmax(np.abs(data)))
        data.plot(
            ax=ax,
            #x="lon",
            #y="lat",
            x="x",
            y="y",
            cmap="RdBu_r",
            vmin=-limit,
            vmax=limit,
        )
    else:
        data.plot(
            ax=ax,
            x="x",
            y="y",
            cmap="viridis",
        )

    ax.set_title(exp)
    ax.set_aspect("equal")

plt.show()

In [ ]:
experiments = ["1D", "3D", "3D-1D", "1DCRE", "3DCRE", "CS"]

# Select one time step and remove the phy dimension of length 1
field = ds["edir"].isel(time=52).squeeze()

fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)

for ax, exp in zip(axes.flat, experiments):
    data = field.sel(exp=exp)

    # Difference/CRE fields: diverging colour scale centred at zero
    if exp in ["3D-1D", "3DCRE", "1DCRE"]:
        limit = float(np.nanmax(np.abs(data)))
        data.plot(
            ax=ax,
            #x="lon",
            #y="lat",
            x="x",
            y="y",
            cmap="RdBu_r",
            vmin=-limit,
            vmax=limit,
        )
    else:
        data.plot(
            ax=ax,
            x="x",
            y="y",
            cmap="viridis",
        )

    ax.set_title(exp)
    ax.set_aspect("equal")

plt.show()

In [ ]:
experiments = ["1D", "3D", "3D-1D", "1DCRE", "3DCRE", "CS"]

# Select one time step and remove the phy dimension of length 1
field = ds["edn"].isel(time=52).squeeze()

fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)

for ax, exp in zip(axes.flat, experiments):
    data = field.sel(exp=exp)

    # Difference/CRE fields: diverging colour scale centred at zero
    if exp in ["3D-1D", "3DCRE", "1DCRE"]:
        limit = float(np.nanmax(np.abs(data)))
        data.plot(
            ax=ax,
            #x="lon",
            #y="lat",
            x="x",
            y="y",
            cmap="RdBu_r",
            vmin=-limit,
            vmax=limit,
        )
    else:
        data.plot(
            ax=ax,
            x="x",
            y="y",
            cmap="viridis",
        )

    ax.set_title(exp)
    ax.set_aspect("equal")

plt.show()

# Gauss filter

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

In [ ]:
time_index = 52

scene = ds.isel(time=time_index).squeeze()

dx = float(np.mean(np.diff(scene["x"].values)))
dy = float(np.mean(np.diff(scene["y"].values)))


print("Time:", scene.time.values)
print("Dimensions:", scene.sizes)
print(f"dx = {dx:.1f} m")
print(f"dy = {dy:.1f} m")

In [ ]:
eglo_1d = scene["eglo"].sel(exp="1D")
edir_1d = scene["edir"].sel(exp="1D")
edn_1d  = scene["edn"].sel(exp="1D")

eglo_3d = scene["eglo"].sel(exp="3D")
edir_3d = scene["edir"].sel(exp="3D")
edn_3d  = scene["edn"].sel(exp="3D")


In [ ]:
sigma_pixel = 2  # sigma in 4 pixels 

sigma_y = sigma_pixel
sigma_x = sigma_pixel

sigma_m = sigma_pixel * (dx+dy)/2 # sigma in meter

edn_1d_filtered = edn_1d.copy(
    data=gaussian_filter(
        edn_1d.values,
        sigma=(sigma_y, sigma_x),
        mode="wrap",
        #mode="reflect",
    )
)

eglo_filtered = edir_1d + edn_1d_filtered

- mode="wrap" :  periodic horizontal domain
- mode="reflect" : non periodic

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9),
    constrained_layout=True,
)

fields = [
    (edn_1d, "1D diffuse"),
    (edn_1d_filtered, f"Filtered 1D diffuse\nσ = {sigma_m:.0f} m / / {sigma_pixel:.0f} pixel"),
    (edn_3d, "3D diffuse"),
    (eglo_1d, "1D global"),
    (eglo_filtered, "Filtered global"),
    (eglo_3d, "3D global"),
]

# Use common scales within each row
diffuse_min = min(
    float(edn_1d.min()),
    float(edn_1d_filtered.min()),
    float(edn_3d.min()),
)
diffuse_max = max(
    float(edn_1d.max()),
    float(edn_1d_filtered.max()),
    float(edn_3d.max()),
)

global_min = min(
    float(eglo_1d.min()),
    float(eglo_filtered.min()),
    float(eglo_3d.min()),
)
global_max = max(
    float(eglo_1d.max()),
    float(eglo_filtered.max()),
    float(eglo_3d.max()),
)

for i, (ax, (field, title)) in enumerate(zip(axes.flat, fields)):
    if i < 3:
        vmin, vmax = diffuse_min, diffuse_max
    else:
        vmin, vmax = global_min, global_max

    field.plot(
        ax=ax,
        x="x",
        y="y",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        cbar_kwargs={"label": "W m$^{-2}$"},
    )

    ax.set_title(title)
    ax.set_aspect("equal")

plt.show()

In [ ]:


vmin = min(
    float(edn_1d.min()),
    float(edn_1d_filtered.min()),
    float(edn_3d.min()),
)

vmax = max(
    float(edn_1d.max()),
    float(edn_1d_filtered.max()),
    float(edn_3d.max()),
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True,)

fields = [
    (edn_1d, "1D diffuse"),
    (edn_1d_filtered, f"Filtered 1D diffuse\nσ = {sigma_m:.0f} m " f"({sigma_pixel:.2f} pixels)"),
    (edn_3d, "3D diffuse"),
]

for ax, (field, title) in zip(axes, fields):
    field.plot(
        ax=ax,
        x="x",
        y="y",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        cbar_kwargs={"label": "W m$^{-2}$"},
    )

    ax.set_title(title)
    ax.set_aspect("equal")

plt.show()

In [ ]:


vmin = min(
    float(eglo_1d.min()),
    float(eglo_filtered.min()),
    float(eglo_3d.min()),
)

vmax = max(
    float(eglo_1d.max()),
    float(eglo_filtered.max()),
    float(eglo_3d.max()),
)

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 5),
    constrained_layout=True,
)

fields = [
    (eglo_1d, "1D global"),
    (eglo_filtered, f"Filtered global\nσ = {sigma_m:.0f} m " f"({sigma_pixel:.2f} pixels)"),
    (eglo_3d, "3D global"),
]

for ax, (field, title) in zip(axes, fields):
    field.plot(
        ax=ax,
        x="x",
        y="y",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        cbar_kwargs={"label": "W m$^{-2}$"},
    )

    ax.set_title(title)
    ax.set_aspect("equal")

plt.show()

In [ ]:
print("1D diffuse mean:",float(edn_1d.mean()))

print("Filtered diffuse mean:",float(edn_1d_filtered.mean()))

print("Mean difference:",float(edn_1d_filtered.mean() - edn_1d.mean()))

In [ ]:
print("1D global mean:",float(eglo_1d.mean()))

print("Filtered global mean:",float(eglo_filtered.mean()))

print("Mean difference:",float(eglo_filtered.mean() - eglo_1d.mean()))

## Matching STD for global field

- matching the spatial standard deviation of filtered 1D **global** irradiance to 3D dimensional **global** irradiance

In [ ]:
sigma_values_m = np.arange(1500, 3501, 50)

std_3d = float(eglo_3d.std())

std_filtered = []

for sigma_m in sigma_values_m:
    sigma_pixel = sigma_m / dx

    edn_1d_filtered = gaussian_filter(
        edn_1d.values,
        sigma=sigma_pixel,
        mode="wrap",
        #mode="reflect",
    )

    eglo_filtered = edir_1d + edn_1d_filtered
    std_filtered.append(np.std(eglo_filtered))

std_filtered = np.array(std_filtered)

best_index = np.argmin(
    np.abs(std_filtered - std_3d)
)

sigma_opt_m = sigma_values_m[best_index]

print(f"3D diffuse std: {std_3d:.2f} W m-2")
print(f"Optimal sigma:  {sigma_opt_m:.0f} m")
print(f"Filtered std:   {std_filtered[best_index]:.2f} W m-2")

In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    sigma_values_m,
    std_filtered,
    marker="o",
)

plt.axhline(
    std_3d,
    linestyle="--",
    label="3D diffuse std",
)

plt.axvline(
    sigma_opt_m,
    linestyle=":",
    label=f"σ_opt = {sigma_opt_m:.0f} m",
)

plt.xlabel("Gaussian filter width σ [m]")
plt.ylabel("Std. dev. diffuse irradiance [W m$^{-2}$]")
plt.legend()
plt.show()

In [ ]:
sigma_opt_pixel = sigma_opt_m / dx

edn_opt_filtered = edn_1d.copy(
    data=gaussian_filter(
        edn_1d.values,
        sigma=sigma_opt_pixel,
        mode="wrap",
        #mode="reflect",
    )
)

# Reconstruct global irradiance
eglo_opt_filtered = edir_1d + edn_opt_filtered

In [ ]:
print(f"1D diffuse std:       {float(edn_1d.std()):.2f} W m-2")
print(f"Filtered diffuse std: {float(edn_opt_filtered.std()):.2f} W m-2")
print(f"3D diffuse std:       {float(edn_3d.std()):.2f} W m-2")
print(f"Optimal sigma:        {sigma_opt_m:.0f} m")

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9),
    constrained_layout=True,
)

fields = [
    (edn_1d, "1D diffuse"),
    (
        edn_opt_filtered,
        f"Filtered 1D diffuse\nσ_opt = {sigma_opt_m:.0f} m "
        f"({sigma_opt_pixel:.2f} pixels)"
    ),
    (edn_3d, "3D diffuse"),
    (eglo_1d, "1D global"),
    (eglo_opt_filtered, "Filtered global"),
    (eglo_3d, "3D global"),
]

# Same color scale within each row
diffuse_min = min(
    float(edn_1d.min()),
    float(edn_opt_filtered.min()),
    float(edn_3d.min()),
)

diffuse_max = max(
    float(edn_1d.max()),
    float(edn_opt_filtered.max()),
    float(edn_3d.max()),
)

global_min = min(
    float(eglo_1d.min()),
    float(eglo_opt_filtered.min()),
    float(eglo_3d.min()),
)

global_max = max(
    float(eglo_1d.max()),
    float(eglo_opt_filtered.max()),
    float(eglo_3d.max()),
)

for i, (ax, (field, title)) in enumerate(zip(axes.flat, fields)):

    if i < 3:
        vmin, vmax = diffuse_min, diffuse_max
    else:
        vmin, vmax = global_min, global_max

    field.plot(
        ax=ax,
        x="x",
        y="y",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        cbar_kwargs={"label": "W m$^{-2}$"},
    )

    ax.set_title(title)
    ax.set_aspect("equal")

fig.suptitle(
    f"{np.datetime_as_string(scene.time.values, unit='m')}   "
    f"|   optimal σ = {sigma_opt_m:.0f} m",
    fontsize=14,
)

plt.show()

## Matching STD for diffuse field

- matching the spatial standard deviation of filtered 1D **diffuse** irradiance to 3D dimensional **diffuse** irradiance

In [ ]:
rmse_1d = np.sqrt(((eglo_1d) ** 2).mean())

rmse_1d_filtered = np.sqrt(((eglo_filtered) ** 2).mean())

rmse_3d = np.sqrt(((eglo_3d) ** 2).mean())

print(f"1D RMSE:       {float(rmse_1d):.2f} W m-2")
print(f"1D Filtered : {float(rmse_1d_filtered):.2f} W m-2")
print(f"3D RMSE:       {float(rmse_3d):.2f} W m-2")

In [ ]:
std_1d = float(edn_1d.std())
std_filtered = float(edn_1d_filtered.std())
std_3d = float(edn_3d.std())

print(f"1D diffuse std:       {std_1d:.2f} W m-2")
print(f"Filtered diffuse std: {std_filtered:.2f} W m-2")
print(f"3D diffuse std:       {std_3d:.2f} W m-2")

In [ ]:
sigma_values_m = np.arange(0, 1501, 50)

std_3d = float(edn_3d.std())

std_filtered = []

for sigma_m in sigma_values_m:
    sigma_pixel = sigma_m / dx

    filtered = gaussian_filter(
        edn_1d.values,
        sigma=sigma_pixel,
        mode="wrap",
        #mode="reflect",
    )

    std_filtered.append(np.std(filtered))

std_filtered = np.array(std_filtered)

best_index = np.argmin(
    np.abs(std_filtered - std_3d)
)

sigma_opt_m = sigma_values_m[best_index]

print(f"3D diffuse std: {std_3d:.2f} W m-2")
print(f"Optimal sigma:  {sigma_opt_m:.0f} m")
print(f"Filtered std:   {std_filtered[best_index]:.2f} W m-2")

In [ ]:
plt.figure(figsize=(7, 5))

plt.plot(
    sigma_values_m,
    std_filtered,
    marker="o",
)

plt.axhline(
    std_3d,
    linestyle="--",
    label="3D diffuse std",
)

plt.axvline(
    sigma_opt_m,
    linestyle=":",
    label=f"σ_opt = {sigma_opt_m:.0f} m",
)

plt.xlabel("Gaussian filter width σ [m]")
plt.ylabel("Std. dev. diffuse irradiance [W m$^{-2}$]")
plt.legend()
plt.show()

In [ ]:
sigma_opt_pixel = sigma_opt_m / dx

edn_opt_filtered = edn_1d.copy(
    data=gaussian_filter(
        edn_1d.values,
        sigma=sigma_opt_pixel,
        mode="wrap",
        #mode="reflect",
    )
)

# Reconstruct global irradiance
eglo_opt_filtered = edir_1d + edn_opt_filtered

In [ ]:
print(f"1D diffuse std:       {float(edn_1d.std()):.2f} W m-2")
print(f"Filtered diffuse std: {float(edn_opt_filtered.std()):.2f} W m-2")
print(f"3D diffuse std:       {float(edn_3d.std()):.2f} W m-2")
print(f"Optimal sigma:        {sigma_opt_m:.0f} m")

In [ ]:
fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 9),
    constrained_layout=True,
)

fields = [
    (edn_1d, "1D diffuse"),
    (
        edn_opt_filtered,
        f"Filtered 1D diffuse\nσ_opt = {sigma_opt_m:.0f} m "
        f"({sigma_opt_pixel:.2f} pixels)"
    ),
    (edn_3d, "3D diffuse"),
    (eglo_1d, "1D global"),
    (eglo_opt_filtered, "Filtered global"),
    (eglo_3d, "3D global"),
]

# Same color scale within each row
diffuse_min = min(
    float(edn_1d.min()),
    float(edn_opt_filtered.min()),
    float(edn_3d.min()),
)

diffuse_max = max(
    float(edn_1d.max()),
    float(edn_opt_filtered.max()),
    float(edn_3d.max()),
)

global_min = min(
    float(eglo_1d.min()),
    float(eglo_opt_filtered.min()),
    float(eglo_3d.min()),
)

global_max = max(
    float(eglo_1d.max()),
    float(eglo_opt_filtered.max()),
    float(eglo_3d.max()),
)

for i, (ax, (field, title)) in enumerate(zip(axes.flat, fields)):

    if i < 3:
        vmin, vmax = diffuse_min, diffuse_max
    else:
        vmin, vmax = global_min, global_max

    field.plot(
        ax=ax,
        x="x",
        y="y",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        cbar_kwargs={"label": "W m$^{-2}$"},
    )

    ax.set_title(title)
    ax.set_aspect("equal")

fig.suptitle(
    f"{np.datetime_as_string(scene.time.values, unit='m')}   "
    f"|   optimal σ = {sigma_opt_m:.0f} m",
    fontsize=14,
)

plt.show()

### Annalysis for after choosing optimal sigma for diffuse field STD

#### PDF

In [ ]:
glo_1d = eglo_1d.values.ravel()
glo_filtered = eglo_opt_filtered.values.ravel()
glo_3d = eglo_3d.values.ravel()

In [ ]:
plt.figure(figsize=(8, 5))

bins = np.linspace(
    min(glo_1d.min(), glo_filtered.min(), glo_3d.min()),
    max(glo_1d.max(), glo_filtered.max(), glo_3d.max()),
    60,
)

plt.hist(
    glo_1d,
    bins=bins,
    density=True,
    histtype="step",
    linewidth=2,
    label="1D",
)

plt.hist(
    glo_filtered,
    bins=bins,
    density=True,
    histtype="step",
    linewidth=2,
    label=f"Filtered 1D (σ={sigma_opt_m:.0f} m)",
)

plt.hist(
    glo_3d,
    bins=bins,
    density=True,
    histtype="step",
    linewidth=2,
    label="3D",
)

plt.xlabel("Global irradiance [W m$^{-2}$]")
plt.ylabel("Probability density")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:

# Flatten fields
glo_1d = eglo_1d.values.ravel()
glo_f  = eglo_opt_filtered.values.ravel()
glo_3d = eglo_3d.values.ravel()

dif_1d = edn_1d.values.ravel()
dif_f  = edn_opt_filtered.values.ravel()
dif_3d = edn_3d.values.ravel()

dir_1d = edir_1d.values.ravel()
dir_3d = edir_3d.values.ravel()

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

# ---------- Global ----------
bins_glo = np.linspace(
    min(glo_1d.min(), glo_f.min(), glo_3d.min()),
    max(glo_1d.max(), glo_f.max(), glo_3d.max()),
    60
)

axes[0].hist(glo_1d, bins=bins_glo, density=True, histtype="step", linewidth=2, label="1D")
axes[0].hist(glo_f,  bins=bins_glo, density=True, histtype="step", linewidth=2,
             label=f"Filtered 1D (σ={sigma_opt_m:.0f} m)")
axes[0].hist(glo_3d, bins=bins_glo, density=True, histtype="step", linewidth=2, label="3D")

axes[0].set_title("Global irradiance")
axes[0].set_xlabel("W m$^{-2}$")
axes[0].set_ylabel("Probability density")
axes[0].grid(alpha=0.3)
axes[0].legend(loc="upper left")

# ---------- Direct ----------
bins_dir = np.linspace(
    min(dir_1d.min(), dir_3d.min()),
    max(dir_1d.max(), dir_3d.max()),
    60
)

axes[1].hist(dir_1d, bins=bins_dir, density=True, histtype="step", linewidth=2, label="1D")
axes[1].hist(dir_3d, bins=bins_dir, density=True, histtype="step", linewidth=2, label="3D")

axes[1].set_title("Direct irradiance")
axes[1].set_xlabel("W m$^{-2}$")
axes[1].set_ylabel("Probability density")
axes[1].grid(alpha=0.3)
axes[1].legend()

# ---------- Diffuse ----------
bins_dif = np.linspace(
    min(dif_1d.min(), dif_f.min(), dif_3d.min()),
    max(dif_1d.max(), dif_f.max(), dif_3d.max()),
    60
)

axes[2].hist(dif_1d, bins=bins_dif, density=True, histtype="step", linewidth=2, label="1D")
axes[2].hist(dif_f,  bins=bins_dif, density=True, histtype="step", linewidth=2,
             label=f"Filtered 1D (σ={sigma_opt_m:.0f} m)")
axes[2].hist(dif_3d, bins=bins_dif, density=True, histtype="step", linewidth=2, label="3D")

axes[2].set_title("Diffuse irradiance")
axes[2].set_xlabel("W m$^{-2}$")
axes[2].set_ylabel("Probability density")
axes[2].grid(alpha=0.3)
axes[2].legend()



fig.suptitle(
    f"PDF comparison at {np.datetime_as_string(scene.time.values, unit='m')}",
    fontsize=14
)

plt.show()

#### Mean and standard deviation

In [ ]:
print("                        Mean        Std")
print("----------------------------------------")

print(
    f"1D glo:              "
    f"{float(eglo_1d.mean()):8.2f}   "
    f"{float(eglo_1d.std()):8.2f}"
)

print(
    f"Filtered 1D glo:     "
    f"{float(eglo_opt_filtered.mean()):8.2f}   "
    f"{float(eglo_opt_filtered.std()):8.2f}"
)

print(
    f"3D glo:              "
    f"{float(eglo_3d.mean()):8.2f}   "
    f"{float(eglo_3d.std()):8.2f}"
)

print()
print(f"mean: domain-average amount of global irradiance") 
print(f"std : overall spatial variability across the scene")

#### Difference maps

In [ ]:
#diff_1d = eglo_1d - eglo_3d
#diff_filtered = eglo_opt_filtered - eglo_3d

diff_1d = edn_1d - edn_3d
diff_filtered = edn_opt_filtered - edn_3d

In [ ]:
limit = max(
    float(np.abs(diff_1d).max()),
    float(np.abs(diff_filtered).max()),
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5),
    constrained_layout=True,
)

diff_1d.plot(
    ax=axes[0],
    x="x",
    y="y",
    cmap="RdBu_r",
    vmin=-limit,
    vmax=limit,
    cbar_kwargs={"label": "Difference [W m$^{-2}$]"},
)

axes[0].set_title("1D − 3D")
axes[0].set_aspect("equal")


diff_filtered.plot(
    ax=axes[1],
    x="x",
    y="y",
    cmap="RdBu_r",
    vmin=-limit,
    vmax=limit,
    cbar_kwargs={"label": "Difference [W m$^{-2}$]"},
)

axes[1].set_title("Filtered 1D − 3D")
axes[1].set_aspect("equal")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Difference fields
diff_dif_1d = edn_1d - edn_3d
diff_dif_f  = edn_opt_filtered - edn_3d

diff_glo_1d = eglo_1d - eglo_3d
diff_glo_f  = eglo_opt_filtered - eglo_3d

# Symmetric color limits per row
lim_dif = max(
    float(np.abs(diff_dif_1d).max()),
    float(np.abs(diff_dif_f).max())
)

lim_glo = max(
    float(np.abs(diff_glo_1d).max()),
    float(np.abs(diff_glo_f).max())
)

fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

# ---------- Top row: Diffuse ----------
diff_dif_1d.plot(
    ax=axes[0, 0],
    x="x",
    y="y",
    cmap="RdBu_r",
    vmin=-lim_dif,
    vmax=lim_dif,
    cbar_kwargs={"label": "Difference [W m$^{-2}$]"},
)
axes[0, 0].set_title("Diffuse: 1D − 3D")
axes[0, 0].set_aspect("equal")

diff_dif_f.plot(
    ax=axes[0, 1],
    x="x",
    y="y",
    cmap="RdBu_r",
    vmin=-lim_dif,
    vmax=lim_dif,
    cbar_kwargs={"label": "Difference [W m$^{-2}$]"},
)
axes[0, 1].set_title("Diffuse: Filtered 1D − 3D")
axes[0, 1].set_aspect("equal")

# ---------- Bottom row: Global ----------
diff_glo_1d.plot(
    ax=axes[1, 0],
    x="x",
    y="y",
    cmap="RdBu_r",
    vmin=-lim_glo,
    vmax=lim_glo,
    cbar_kwargs={"label": "Difference [W m$^{-2}$]"},
)
axes[1, 0].set_title("Global: 1D − 3D")
axes[1, 0].set_aspect("equal")

diff_glo_f.plot(
    ax=axes[1, 1],
    x="x",
    y="y",
    cmap="RdBu_r",
    vmin=-lim_glo,
    vmax=lim_glo,
    cbar_kwargs={"label": "Difference [W m$^{-2}$]"},
)
axes[1, 1].set_title("Global: Filtered 1D − 3D")
axes[1, 1].set_aspect("equal")

fig.suptitle(
    f"Difference maps at {np.datetime_as_string(scene.time.values, unit='m')}   "
    f"|   σ_opt = {sigma_opt_m:.0f} m",
    fontsize=14,
)

plt.show()

## Shifting direct radiation manually do fix displacement due to solar geometry

In [ ]:
shift_x = 4  
shift_y = 5 

edir_1d_shifted = edir_1d.copy(
    data=np.roll(
        edir_1d.values,
        shift=(shift_y, shift_x),
        axis=(0, 1)
    )
)

print(f"x shift: {shift_x * dx:.0f} m")
print(f"y shift: {shift_y * dy:.0f} m")

In [ ]:
eglo_shifted_filtered = (
    edir_1d_shifted
    + edn_opt_filtered
)

In [ ]:
fig, axes = plt.subplots(
    3,
    3,
    figsize=(15, 13),
    constrained_layout=True,
)

fields = [
    # Row 1: direct
    (edir_1d, "1D direct"),
    (edir_1d_shifted, "Shifted 1D direct"),
    (edir_3d, "3D direct"),

    # Row 2: diffuse
    (edn_1d, "1D diffuse"),
    (
        edn_opt_filtered,
        f"Filtered 1D diffuse\nσ_opt = {sigma_opt_m:.0f} m "
        f"({sigma_opt_pixel:.2f} pixels)"
    ),
    (edn_3d, "3D diffuse"),

    # Row 3: global
    (eglo_1d, "1D global"),
    (eglo_shifted_filtered, "Shifted + filtered global"),
    (eglo_3d, "3D global"),
]

# Same color scale within each row
direct_min = min(
    float(edir_1d.min()),
    float(edir_1d_shifted.min()),
    float(edir_3d.min()),
)
direct_max = max(
    float(edir_1d.max()),
    float(edir_1d_shifted.max()),
    float(edir_3d.max()),
)

diffuse_min = min(
    float(edn_1d.min()),
    float(edn_opt_filtered.min()),
    float(edn_3d.min()),
)
diffuse_max = max(
    float(edn_1d.max()),
    float(edn_opt_filtered.max()),
    float(edn_3d.max()),
)

global_min = min(
    float(eglo_1d.min()),
    float(eglo_shifted_filtered.min()),
    float(eglo_3d.min()),
)
global_max = max(
    float(eglo_1d.max()),
    float(eglo_shifted_filtered.max()),
    float(eglo_3d.max()),
)

for i, (ax, (field, title)) in enumerate(zip(axes.flat, fields)):

    if i < 3:
        vmin, vmax = direct_min, direct_max
    elif i < 6:
        vmin, vmax = diffuse_min, diffuse_max
    else:
        vmin, vmax = global_min, global_max

    field.plot(
        ax=ax,
        x="x",
        y="y",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        cbar_kwargs={"label": "W m$^{-2}$"},
    )

    ax.set_title(title)
    ax.set_aspect("equal")

fig.suptitle(
    f"{np.datetime_as_string(scene.time.values, unit='m')}   "
    f"|   σ_opt = {sigma_opt_m:.0f} m",
    fontsize=14,
)

plt.show()

In [ ]:
print("                              Mean        Std       RMSE")
print("----------------------------------------------------------")

print(
    f"1D glo:                    "
    f"{float(eglo_1d.mean()):8.2f}   "
    f"{float(eglo_1d.std()):8.2f}   "
    f"{float(np.sqrt(((eglo_1d - eglo_3d) ** 2).mean())):8.2f}"
)

print(
    f"Filtered 1D glo:           "
    f"{float(eglo_opt_filtered.mean()):8.2f}   "
    f"{float(eglo_opt_filtered.std()):8.2f}   "
    f"{float(np.sqrt(((eglo_opt_filtered - eglo_3d) ** 2).mean())):8.2f}"
)

print(
    f"Shifted + filtered glo:    "
    f"{float(eglo_shifted_filtered.mean()):8.2f}   "
    f"{float(eglo_shifted_filtered.std()):8.2f}   "
    f"{float(np.sqrt(((eglo_shifted_filtered - eglo_3d) ** 2).mean())):8.2f}"
)

print(
    f"3D glo:                    "
    f"{float(eglo_3d.mean()):8.2f}   "
    f"{float(eglo_3d.std()):8.2f}   "
    f"{0.0:8.2f}"
)

print()
print("mean: domain-average amount of global irradiance")
print("std : overall spatial variability across the scene")
print("RMSE: pixel-wise deviation from the 3D reference field")

why is RMSE for filtered 1d glo so big?
- RMSE: does each individual pixel contain approximately the same irradiance as the corresponding 3D pixel?
- apparently the answer is worse after filtering, because the direct cloud shadows are still in the wrong locations.

In [ ]:

# Difference fields
diff_dif_1d = edn_1d - edn_3d
diff_dif_mod = edn_opt_filtered - edn_3d

diff_glo_1d = eglo_1d - eglo_3d
diff_glo_mod = eglo_shifted_filtered - eglo_3d

# Symmetric color limits per row
lim_dif = max(
    float(np.abs(diff_dif_1d).max()),
    float(np.abs(diff_dif_mod).max())
)

lim_glo = max(
    float(np.abs(diff_glo_1d).max()),
    float(np.abs(diff_glo_mod).max())
)

fig, axes = plt.subplots(2, 2, figsize=(12, 10), constrained_layout=True)

# ---------- Top row: Diffuse ----------
diff_dif_1d.plot(
    ax=axes[0, 0],
    x="x",
    y="y",
    cmap="RdBu_r",
    vmin=-lim_dif,
    vmax=lim_dif,
    cbar_kwargs={"label": "Difference [W m$^{-2}$]"},
)
axes[0, 0].set_title("Diffuse: 1D − 3D")
axes[0, 0].set_aspect("equal")

diff_dif_mod.plot(
    ax=axes[0, 1],
    x="x",
    y="y",
    cmap="RdBu_r",
    vmin=-lim_dif,
    vmax=lim_dif,
    cbar_kwargs={"label": "Difference [W m$^{-2}$]"},
)
axes[0, 1].set_title("Diffuse: Filtered 1D − 3D")
axes[0, 1].set_aspect("equal")

# ---------- Bottom row: Global ----------
diff_glo_1d.plot(
    ax=axes[1, 0],
    x="x",
    y="y",
    cmap="RdBu_r",
    vmin=-lim_glo,
    vmax=lim_glo,
    cbar_kwargs={"label": "Difference [W m$^{-2}$]"},
)
axes[1, 0].set_title("Global: 1D − 3D")
axes[1, 0].set_aspect("equal")

diff_glo_mod.plot(
    ax=axes[1, 1],
    x="x",
    y="y",
    cmap="RdBu_r",
    vmin=-lim_glo,
    vmax=lim_glo,
    cbar_kwargs={"label": "Difference [W m$^{-2}$]"},
)
axes[1, 1].set_title("Global: Shifted + filtered − 3D")
axes[1, 1].set_aspect("equal")

fig.suptitle(
    f"Difference maps at {np.datetime_as_string(scene.time.values, unit='m')}   "
    f"|   σ_opt = {sigma_opt_m:.0f} m",
    fontsize=14,
)

plt.show()

In [ ]:
# Difference fields
diff_glo_filtered = eglo_opt_filtered - eglo_3d
diff_glo_shifted_filtered = eglo_shifted_filtered - eglo_3d

# Common symmetric color scale
lim_glo = max(
    float(np.abs(diff_glo_filtered).max()),
    float(np.abs(diff_glo_shifted_filtered).max())
)

fig, axes = plt.subplots(
    1, 2,
    figsize=(12, 5),
    constrained_layout=True
)

# Filtered only
diff_glo_filtered.plot(
    ax=axes[0],
    x="x",
    y="y",
    cmap="RdBu_r",
    vmin=-lim_glo,
    vmax=lim_glo,
    cbar_kwargs={"label": "Difference [W m$^{-2}$]"},
)

axes[0].set_title("Global: Filtered 1D − 3D")
axes[0].set_aspect("equal")


# Shifted + filtered
diff_glo_shifted_filtered.plot(
    ax=axes[1],
    x="x",
    y="y",
    cmap="RdBu_r",
    vmin=-lim_glo,
    vmax=lim_glo,
    cbar_kwargs={"label": "Difference [W m$^{-2}$]"},
)

axes[1].set_title("Global: Shifted + Filtered 1D − 3D")
axes[1].set_aspect("equal")


fig.suptitle(
    f"{np.datetime_as_string(scene.time.values, unit='m')}   "
    f"|   σ_opt = {sigma_opt_m:.0f} m",
    fontsize=14
)

plt.show()

In [ ]:
bias_filtered = float((eglo_opt_filtered - eglo_3d).mean())
bias_shifted = float((eglo_shifted_filtered - eglo_3d).mean())

mae_filtered = float(np.abs(eglo_opt_filtered - eglo_3d).mean())
mae_shifted = float(np.abs(eglo_shifted_filtered - eglo_3d).mean())

rmse_filtered = float(
    np.sqrt(((eglo_opt_filtered - eglo_3d) ** 2).mean())
)

rmse_shifted = float(
    np.sqrt(((eglo_shifted_filtered - eglo_3d) ** 2).mean())
)

print("                           Bias       MAE       RMSE")
print("----------------------------------------------------")

print(
    f"Filtered 1D:          "
    f"{bias_filtered:8.2f}  "
    f"{mae_filtered:8.2f}  "
    f"{rmse_filtered:8.2f}"
)

print(
    f"Shifted + filtered:   "
    f"{bias_shifted:8.2f}  "
    f"{mae_shifted:8.2f}  "
    f"{rmse_shifted:8.2f}"
)

In [ ]:
corr_1d = np.corrcoef(
    eglo_1d.values.ravel(),
    eglo_3d.values.ravel()
)[0, 1]

corr_filtered = np.corrcoef(
    eglo_opt_filtered.values.ravel(),
    eglo_3d.values.ravel()
)[0, 1]

corr_shifted_filtered = np.corrcoef(
    eglo_shifted_filtered.values.ravel(),
    eglo_3d.values.ravel()
)[0, 1]

print(f"1D:                 {corr_1d:.3f}")
print(f"Filtered 1D:        {corr_filtered:.3f}")
print(f"Shifted + filtered: {corr_shifted_filtered:.3f}")

In [ ]:
# Direct
corr_dir = np.corrcoef(
    edir_1d.values.ravel(),
    edir_3d.values.ravel()
)[0, 1]

corr_dir_shifted = np.corrcoef(
    edir_1d_shifted.values.ravel(),
    edir_3d.values.ravel()
)[0, 1]

# Diffuse
corr_dif = np.corrcoef(
    edn_1d.values.ravel(),
    edn_3d.values.ravel()
)[0, 1]

corr_dif_filtered = np.corrcoef(
    edn_opt_filtered.values.ravel(),
    edn_3d.values.ravel()
)[0, 1]

print("Direct:")
print(f"1D:       {corr_dir:.3f}")
print(f"Shifted:  {corr_dir_shifted:.3f}")

print()
print("Diffuse:")
print(f"1D:       {corr_dif:.3f}")
print(f"Filtered: {corr_dif_filtered:.3f}")

In [ ]:
# Calculate metrics
mean_1d = float(eglo_1d.mean())
std_1d = float(eglo_1d.std())
rmse_1d = float(
    np.sqrt(((eglo_1d - eglo_3d) ** 2).mean())
)
corr_1d = np.corrcoef(
    eglo_1d.values.ravel(),
    eglo_3d.values.ravel()
)[0, 1]

mean_filtered = float(eglo_opt_filtered.mean())
std_filtered = float(eglo_opt_filtered.std())
rmse_filtered = float(
    np.sqrt(((eglo_opt_filtered - eglo_3d) ** 2).mean())
)
corr_filtered = np.corrcoef(
    eglo_opt_filtered.values.ravel(),
    eglo_3d.values.ravel()
)[0, 1]

mean_shifted = float(eglo_shifted_filtered.mean())
std_shifted = float(eglo_shifted_filtered.std())
rmse_shifted = float(
    np.sqrt(((eglo_shifted_filtered - eglo_3d) ** 2).mean())
)
corr_shifted = np.corrcoef(
    eglo_shifted_filtered.values.ravel(),
    eglo_3d.values.ravel()
)[0, 1]

mean_3d = float(eglo_3d.mean())
std_3d_glo = float(eglo_3d.std())


# Print table
print(
    f"{'':28s}"
    f"{'Mean':>10s}"
    f"{'Std':>10s}"
    f"{'RMSE':>10s}"
    f"{'Corr':>10s}"
)

print("-" * 68)

print(
    f"{'1D global':28s}"
    f"{mean_1d:10.2f}"
    f"{std_1d:10.2f}"
    f"{rmse_1d:10.2f}"
    f"{corr_1d:10.3f}"
)

print(
    f"{'Filtered 1D global':28s}"
    f"{mean_filtered:10.2f}"
    f"{std_filtered:10.2f}"
    f"{rmse_filtered:10.2f}"
    f"{corr_filtered:10.3f}"
)

print(
    f"{'Shifted + filtered global':28s}"
    f"{mean_shifted:10.2f}"
    f"{std_shifted:10.2f}"
    f"{rmse_shifted:10.2f}"
    f"{corr_shifted:10.3f}"
)

print(
    f"{'3D global':28s}"
    f"{mean_3d:10.2f}"
    f"{std_3d_glo:10.2f}"
    f"{0.0:10.2f}"
    f"{1.0:10.3f}"
)

print()
print("Mean : domain-average global irradiance [W m-2]")
print("Std  : overall spatial variability [W m-2]")
print("RMSE : pixel-wise deviation from 3D reference [W m-2]")
print("Corr : spatial Pearson correlation with 3D reference")

## Shift Direct AND diffuse filtered field

In [ ]:
# Shift settings
shift_x = 4  
shift_y = 5 

# Shift filtered diffuse field
edn_filtered_shifted = edn_opt_filtered.copy(
    data=np.roll(
        edn_opt_filtered.values,
        shift=(shift_y, shift_x),
        axis=(0, 1)
    )
)

# New global field:
# shifted direct + shifted filtered diffuse
eglo_both_shifted = (
    edir_1d_shifted
    + edn_filtered_shifted
)

In [ ]:
# Helper function
def calc_metrics(field, reference):
    mean = float(field.mean())
    std = float(field.std())

    rmse = float(
        np.sqrt(((field - reference) ** 2).mean())
    )

    corr = np.corrcoef(
        field.values.ravel(),
        reference.values.ravel()
    )[0, 1]

    return mean, std, rmse, corr


# Calculate metrics
metrics_1d = calc_metrics(
    eglo_1d,
    eglo_3d
)

metrics_filtered = calc_metrics(
    eglo_opt_filtered,
    eglo_3d
)

metrics_shifted_filtered = calc_metrics(
    eglo_shifted_filtered,
    eglo_3d
)

metrics_both_shifted = calc_metrics(
    eglo_both_shifted,
    eglo_3d
)

metrics_3d = (
    float(eglo_3d.mean()),
    float(eglo_3d.std()),
    0.0,
    1.0
)


# Print table
print(
    f"{'':32s}"
    f"{'Mean':>10s}"
    f"{'Std':>10s}"
    f"{'RMSE':>10s}"
    f"{'Corr':>10s}"
)

print("-" * 72)

rows = [
    ("1D global", metrics_1d),
    ("Filtered 1D global", metrics_filtered),
    ("Shifted direct + filtered", metrics_shifted_filtered),
    ("Shifted direct + shifted diffuse", metrics_both_shifted),
    ("3D global", metrics_3d),
]

for name, metrics in rows:
    mean, std, rmse, corr = metrics

    print(
        f"{name:32s}"
        f"{mean:10.2f}"
        f"{std:10.2f}"
        f"{rmse:10.2f}"
        f"{corr:10.3f}"
    )

- The direct component is dominated by cloud-shadow geometry, so a rigid displacement can approximate the 3D shadow shift quite well. The diffuse component, however, is affected by horizontal photon transport, illumination of cloud sides, scattering, and smoothing around cloud edges. It is not simply the same field translated by the cloud-shadow displacement.
- So shifting the diffuse field by the same geometrical offset as the direct field is not beneficial.

## Tagesverlauf sigma_opt

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# Sigma search range
sigma_values_m = np.arange(0, 3501, 50)

# Storage
times = []
std_3d_all = []
sigma_opt_all = []
std_filtered_opt_all = []

# Grid spacing
dx = float(np.mean(np.diff(ds["x"].values)))

# Loop over all timesteps
for time_index in range(ds.sizes["time"]):

    scene = ds.isel(time=time_index).squeeze()

    # Diffuse irradiance
    edn_1d = scene["edn"].sel(exp="1D")
    edn_3d = scene["edn"].sel(exp="3D")

    # 3D diffuse spatial standard deviation
    std_3d = float(edn_3d.std())

    std_filtered = []

    # Test all sigma values
    for sigma_m in sigma_values_m:

        sigma_pixel = sigma_m / dx

        edn_filtered = gaussian_filter(
            edn_1d.values,
            sigma=sigma_pixel,
            mode="wrap",
        )

        std_filtered.append(
            np.std(edn_filtered)
        )

    std_filtered = np.array(std_filtered)

    # Sigma giving closest match to 3D std
    best_index = np.argmin(
        np.abs(std_filtered - std_3d)
    )

    sigma_opt_m = sigma_values_m[best_index]

    # Store results
    times.append(scene.time.values)
    std_3d_all.append(std_3d)
    sigma_opt_all.append(sigma_opt_m)
    std_filtered_opt_all.append(
        std_filtered[best_index]
    )


# Convert to arrays
times = np.array(times)
std_3d_all = np.array(std_3d_all)
sigma_opt_all = np.array(sigma_opt_all)
std_filtered_opt_all = np.array(std_filtered_opt_all)

In [ ]:
fig, axes = plt.subplots(
    2, 1,
    figsize=(10, 8),
    sharex=True,
    constrained_layout=True
)

# ------------------------------------------------
# Top: 3D diffuse std and optimally filtered std
# ------------------------------------------------
axes[0].plot(
    times,
    std_3d_all,
    marker="o",
    label="3D diffuse"
)

axes[0].plot(
    times,
    std_filtered_opt_all,
    marker="o",
    linestyle="--",
    label="Filtered 1D diffuse"
)

axes[0].set_ylabel(
    "Spatial std. dev. [W m$^{-2}$]"
)

axes[0].set_title(
    "Daytime variability of diffuse irradiance"
)

axes[0].legend()
axes[0].grid(alpha=0.3)


# ------------------------------------------------
# Bottom: optimal sigma
# ------------------------------------------------
axes[1].plot(
    times,
    sigma_opt_all,
    marker="o"
)

axes[1].set_ylabel(
    "Optimal σ [m]"
)

axes[1].set_xlabel(
    "Time"
)

axes[1].set_title(
    "Optimal Gaussian filter width"
)

axes[1].grid(alpha=0.3)

plt.show()

In [ ]:
print(list(ds.data_vars))


In [ ]:
print(ds["masks"])
print()
print(ds["masks"].attrs)
print()
print("Unique values:")
print(np.unique(ds["masks"].values))

In [ ]:
# Boolean 3D cloud mask
cloud_3d = ds["masks"] > 0

# A horizontal grid point is cloudy if cloud exists
# at any vertical level in that column
cloud_2d = cloud_3d.any(dim="z")

# Fraction of horizontally cloudy grid cells
cloud_cover_all = (
    cloud_2d.mean(dim=("lat", "lon")).values * 100
)

print(cloud_cover_all.shape)
print(f"Cloud cover range: "
      f"{cloud_cover_all.min():.1f}–{cloud_cover_all.max():.1f} %")

In [ ]:
fig, axes = plt.subplots(
    3, 1,
    figsize=(10, 10),
    sharex=True,
    constrained_layout=True
)

# ------------------------------------------------
# 1) Diffuse spatial standard deviation
# ------------------------------------------------
axes[0].plot(
    times,
    std_3d_all,
    marker="o",
    label="3D diffuse"
)

axes[0].plot(
    times,
    std_filtered_opt_all,
    marker="o",
    linestyle="--",
    label="Filtered 1D diffuse"
)

axes[0].set_ylabel(
    "Spatial std. dev.\n[W m$^{-2}$]"
)

axes[0].set_title(
    "Diffuse irradiance variability"
)

axes[0].legend()
axes[0].grid(alpha=0.3)


# ------------------------------------------------
# 2) Optimal Gaussian sigma
# ------------------------------------------------
axes[1].plot(
    times,
    sigma_opt_all,
    marker="o"
)

axes[1].set_ylabel(
    "Optimal σ [m]"
)

axes[1].set_title(
    "Optimal Gaussian filter width"
)

axes[1].grid(alpha=0.3)


# ------------------------------------------------
# 3) Cloud cover
# ------------------------------------------------
axes[2].plot(
    times,
    cloud_cover_all,
    marker="o"
)

axes[2].set_ylabel(
    "Cloud cover [%]"
)

axes[2].set_ylim(0, 100)

axes[2].set_xlabel(
    "Time"
)

axes[2].set_title(
    "Cloud cover"
)

axes[2].grid(alpha=0.3)

plt.show()

In [ ]:
fig, axes = plt.subplots(
    2, 1,
    figsize=(10, 8),
    sharex=True,
    constrained_layout=True
)

# ------------------------------------------------
# Top: diffuse spatial standard deviation
# ------------------------------------------------
axes[0].plot(
    times,
    std_3d_all,
    marker="o",
    label="3D diffuse"
)

axes[0].plot(
    times,
    std_filtered_opt_all,
    marker="o",
    linestyle="--",
    label="Filtered 1D diffuse"
)

axes[0].set_ylabel(
    "Spatial std. dev. [W m$^{-2}$]"
)

axes[0].set_title(
    "Diffuse irradiance variability"
)

axes[0].legend()
axes[0].grid(alpha=0.3)


# ------------------------------------------------
# Bottom: optimal sigma
# ------------------------------------------------
ax_sigma = axes[1]

line_sigma, = ax_sigma.plot(
    times,
    sigma_opt_all,
    marker="o",
    label="Optimal σ"
)

ax_sigma.set_ylabel(
    "Optimal σ [m]"
)

ax_sigma.set_xlabel(
    "Time"
)

ax_sigma.set_title(
    "Optimal Gaussian filter width and cloud cover"
)

ax_sigma.grid(alpha=0.3)


# ------------------------------------------------
# Second y-axis: cloud cover
# ------------------------------------------------
ax_cloud = ax_sigma.twinx()

line_cloud, = ax_cloud.plot(
    times,
    cloud_cover_all,
    marker="s",
    linestyle="--",
    label="Cloud cover",
    color= "red"
)

ax_cloud.set_ylabel(
    "Cloud cover [%]"
)

ax_cloud.set_ylim(0, 100)


# ------------------------------------------------
# Combined legend
# ------------------------------------------------
ax_sigma.legend(
    [line_sigma, line_cloud],
    ["Optimal σ", "Cloud cover"],
    loc="best"
)

plt.show()